# Lab 02 — Advanced HTML and Native Interactivity 🧪
**ORT Web Development | Google Colab**

This lab follows [Lab 01 — HTML Fundamentals](https://colab.research.google.com/github/morbargig/html-practice/blob/main/labs/lab-01-html-fundamentals.ipynb). You will build form controls and page interactions with **HTML only**. Leave out CSS and JavaScript.

**What you'll practice:** attributes, advanced inputs, fieldsets, datalists, details and summary, popovers, dialogs, progress, and meter.

**How to use this notebook**
1. Select **File → Save a copy in Drive** so your work is yours.
2. Run the **Setup** cell once.
3. Edit each `html_...` variable, then run its cell.
4. Run the matching **Preview & Tests** cell to see your page and feedback.
5. Fix failed checks and rerun until everything passes.

**Note:** Popovers and dialogs need a current desktop browser. Colab can check the markup and show a simple preview. Download `student-portal.html` from the final exercise and open it in Chrome to try those controls. For the numbered practice folders, continue with [HTML Practice on GitHub](https://github.com/morbargig/html-practice).

## Setup — Run this cell first
This installs the HTML parser and creates the same preview and test helpers as Lab 01.

In [ ]:
%pip -q install beautifulsoup4
from bs4 import BeautifulSoup
from IPython.display import HTML, display
from html import escape

def preview(source):
    # iframe isolates the rendered student page from Colab's interface
    display(HTML('<iframe sandbox="allow-scripts allow-popups allow-modals" style="width:100%;height:320px;border:1px solid #aaa;border-radius:8px" srcdoc="' +
                 escape(source, quote=True) + '"></iframe>'))

def check(source, tests):
    soup = BeautifulSoup(source, "html.parser")
    passed = 0
    for label, test in tests:
        try:
            ok = bool(test(soup))
        except Exception:
            ok = False
        print(("✅ PASS" if ok else "❌ FIX ") + " — " + label)
        passed += ok
    print(f"\nResult: {passed}/{len(tests)} checks passed")
    return passed == len(tests)

print("Setup complete! Continue to Exercise 1.")

## Exercise 1 — Attributes and Advanced Inputs

Some attributes change what an element does. `title` shows a tooltip. `contenteditable="true"` lets the reader change the text. The `type` of an `<input>` chooses the control: a date, a color, or a range.

**Your task**
- Add a `title` tooltip on the `<h1>`.
- Make the bio paragraph editable with `contenteditable="true"`.
- Mark the name field `required`.
- Use `type="date"` for the birthday.
- Use `type="color"` for the favorite color.
- Use `type="range"` for skill level, and set `min` and `max`.

Keep the existing `id` values so each label stays connected to its control.

In [ ]:
html_1 = '''
<h1>Student Details</h1>
<p>Click here to edit your bio.</p>
<form>
  <label for="student-name">Name:</label>
  <input id="student-name" name="student-name">
  <br>
  <label for="birthday">Birthday:</label>
  <input id="birthday" name="birthday">
  <br>
  <label for="color">Favorite color:</label>
  <input id="color" name="color">
  <br>
  <label for="skill">Skill level:</label>
  <input id="skill" name="skill">
</form>
'''

In [ ]:
preview(html_1)

def input_named(soup, field_id):
    return soup.find("input", id=field_id)

check(html_1, [
    ("The heading has a title tooltip", lambda s: s.h1 is not None and bool(s.h1.get("title", "").strip())),
    ("The bio paragraph is editable", lambda s: any((p.get("contenteditable") or "").lower() == "true" for p in s.find_all("p"))),
    ("The name field is required", lambda s: input_named(s, "student-name") is not None and input_named(s, "student-name").has_attr("required")),
    ("Birthday uses a date input", lambda s: input_named(s, "birthday") is not None and input_named(s, "birthday").get("type") == "date"),
    ("Favorite color uses a color input", lambda s: input_named(s, "color") is not None and input_named(s, "color").get("type") == "color"),
    ("Skill level is a range with min and max", lambda s: input_named(s, "skill") is not None and input_named(s, "skill").get("type") == "range" and input_named(s, "skill").has_attr("min") and input_named(s, "skill").has_attr("max")),
])

## Exercise 2 — Fieldset, Legend, and Datalist

A `<fieldset>` groups related controls. Its `<legend>` names that group.

A `<datalist>` suggests values for a text field. The reader can still type a course that is not in the list. A `<select>` limits the choice to its options.

**Your task**
- Wrap the email field in a `<fieldset>` whose `<legend>` is **Personal Details**.
- Add a second `<fieldset>` whose `<legend>` is **Course Preferences**.
- Inside it, add a text input with `list="courses"`.
- Add `<datalist id="courses">` with options for **HTML**, **CSS**, and **JavaScript**.
- Add a submit button.

In [ ]:
html_2 = '''
<form>
  <label for="email">Email:</label>
  <input id="email" name="email" type="email" required>

  <!-- TODO: Wrap the email field in a fieldset with the legend Personal Details -->
  <!-- TODO: Add a Course Preferences fieldset, a course input, and a datalist -->
  <!-- TODO: Add a submit button -->
</form>
'''

In [ ]:
preview(html_2)
check(html_2, [
    ("Personal Details groups the email field", lambda s: any(
        fs.legend and fs.legend.get_text(" ", strip=True).lower() == "personal details" and fs.find("input", id="email")
        for fs in s.find_all("fieldset")
    )),
    ("Course Preferences is a second fieldset", lambda s: any(
        fs.legend and fs.legend.get_text(" ", strip=True).lower() == "course preferences"
        for fs in s.find_all("fieldset")
    )),
    ("The course field uses the datalist", lambda s: any(inp.get("list") == "courses" for inp in s.find_all("input"))),
    ("The datalist includes HTML, CSS, and JavaScript", lambda s: s.find("datalist", id="courses") is not None and {"html", "css", "javascript"} <= {opt.get("value", "").strip().lower() for opt in s.find("datalist", id="courses").find_all("option")}),
    ("The form has a submit button", lambda s: any(btn.get("type") == "submit" for btn in s.find_all("button"))),
])

## Exercise 3 — Details, Popover, and Dialog

These elements open and close in the browser. You do not need JavaScript.

- `<details>` and `<summary>` make a disclosure. The `open` attribute starts it expanded.
- A button with `popovertarget` toggles the element that has the matching `id` and the `popover` attribute.
- A `<dialog>` is a modal. Open it with `commandfor` set to the dialog `id` and `command="show-modal"`. Close it with `command="close"`.

`command` and `commandfor` need a current version of Chrome. Colab's preview may leave the popover and dialog closed. The checks below read the markup either way.

**Your task**
- Add two `<details>` blocks. Put the `open` attribute on the first one.
- Give each block a `<summary>` and a paragraph.
- Add a help button with `popovertarget="help"` and a `<div id="help" popover>`.
- Add `<dialog id="info-dialog">` with a heading.
- Add an **Open Information** button with `commandfor="info-dialog"` and `command="show-modal"`.
- Inside the dialog, add a **Close** button with `commandfor="info-dialog"` and `command="close"`.

In [ ]:
html_3 = '''
<h1>Student Help Center</h1>
<!-- TODO: Add two details elements. Open the first one. -->
<!-- TODO: Add a help popover and the button that opens it. -->
<!-- TODO: Add a dialog, an open button, and a close button. -->
'''

In [ ]:
preview(html_3)
check(html_3, [
    ("There are at least two details blocks", lambda s: len(s.find_all("details")) >= 2),
    ("One details block starts open", lambda s: any(block.has_attr("open") for block in s.find_all("details"))),
    ("Each details block has a summary and a paragraph", lambda s: len(s.find_all("details")) >= 2 and all(block.summary and block.summary.get_text(strip=True) and block.find("p") and block.find("p").get_text(strip=True) for block in s.find_all("details"))),
    ("A button targets the help popover", lambda s: any(btn.get("popovertarget") == "help" for btn in s.find_all("button"))),
    ("The help popover exists", lambda s: s.find(id="help") is not None and s.find(id="help").has_attr("popover")),
    ("The information dialog exists", lambda s: s.find("dialog", id="info-dialog") is not None),
    ("A button opens the dialog", lambda s: any(btn.get("commandfor") == "info-dialog" and btn.get("command") == "show-modal" for btn in s.find_all("button"))),
    ("The dialog contains a close button", lambda s: s.find("dialog", id="info-dialog") is not None and any(btn.get("commandfor") == "info-dialog" and btn.get("command") == "close" for btn in s.find("dialog", id="info-dialog").find_all("button"))),
])

## Exercise 4 — Progress and Meter

`<progress>` shows how far a task has gone, such as course completion. `<meter>` shows a measurement inside a known range, such as storage used. `low`, `high`, and `optimum` mark the preferred part of that range.

**Your task**
- Set course completion to **45** out of **100** on `<progress id="progress">`.
- Set storage usage to **0.9** on `<meter id="storage">`.
- Give the meter `min="0"`, `max="1"`, `low`, `high`, and `optimum`.
- Keep each label connected with `for` and `id`.

In [ ]:
html_4 = '''
<label for="progress">Course completion:</label>
<progress id="progress" max="100">0%</progress>
<br>
<label for="storage">Storage used:</label>
<meter id="storage" min="0" max="1">0%</meter>
'''

In [ ]:
preview(html_4)
check(html_4, [
    ("Course completion is 45 out of 100", lambda s: s.find("progress", id="progress") is not None and s.find("progress", id="progress").get("value") == "45" and s.find("progress", id="progress").get("max") == "100"),
    ("The completion label matches the progress id", lambda s: s.find("label", attrs={"for": "progress"}) is not None),
    ("Storage usage is 0.9", lambda s: s.find("meter", id="storage") is not None and s.find("meter", id="storage").get("value") in {"0.9", "0.90", ".9"}),
    ("The meter has min, max, low, high, and optimum", lambda s: s.find("meter", id="storage") is not None and all(s.find("meter", id="storage").has_attr(attr) for attr in ("min", "max", "low", "high", "optimum"))),
    ("The storage label matches the meter id", lambda s: s.find("label", attrs={"for": "storage"}) is not None),
])

## Final Challenge — Interactive Student Portal 🚀

Combine this lab into one complete HTML document. The test cell saves it as `student-portal.html`.

**Requirements**
- A doctype, `<html lang="en">`, a character set, and a non-empty `<title>`.
- A registration `<form>` with at least two `<fieldset>` elements, each with a `<legend>`.
- Connected labels for a name field, plus `email`, `date`, `color`, and `range` inputs.
- A `<datalist>` with at least three options, and a text input that uses it.
- Three FAQ items with `<details>` and `<summary>`. One item starts open.
- A help `<button>` that opens an element with the `popover` attribute.
- A `<dialog>` with an open button (`command="show-modal"`) and a close button (`command="close"`).
- A `<progress>` element and a `<meter>` element, each with a `value`.
- An editable bio with `contenteditable="true"`.
- HTML only: no `<style>`, no `<script>`, and no `style` attributes.

**Bonus (not graded):** Add a file input and a second popover.

After the checks pass, open the files panel in Colab, download the page, and try the popover and dialog in Chrome.

In [ ]:
html_final = '''
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Interactive Student Portal</title>
</head>
<body>
  <header><h1>Interactive Student Portal</h1></header>
  <main>
    <!-- TODO: Add an editable bio -->
    <!-- TODO: Add a form with two fieldsets, labels, and advanced inputs -->
    <!-- TODO: Add a datalist for course selection -->
    <!-- TODO: Add progress and meter -->
    <!-- TODO: Add three FAQ details elements, with one open -->
    <!-- TODO: Add a help popover -->
    <!-- TODO: Add a modal dialog with open and close buttons -->
  </main>
  <footer><p>HTML Lab 02</p></footer>
</body>
</html>
'''

In [ ]:
from pathlib import Path
Path("student-portal.html").write_text(html_final, encoding="utf-8")
print("Saved student-portal.html. Download it from the Files panel and open it in Chrome.")
preview(html_final)

def labeled(soup, control):
    control_id = control.get("id", "").strip()
    return bool(control_id) and soup.find("label", attrs={"for": control_id}) is not None

check(html_final, [
    ("Has an HTML5 doctype", lambda s: html_final.lstrip().lower().startswith("<!doctype html>")),
    ("HTML language is English", lambda s: s.html is not None and s.html.get("lang") == "en"),
    ("Has a non-empty title", lambda s: s.title is not None and bool(s.title.get_text(strip=True))),
    ("The form has two labeled fieldsets", lambda s: sum(1 for fs in s.find_all("fieldset") if fs.legend and fs.legend.get_text(strip=True)) >= 2),
    ("Name, email, date, color, and range inputs have labels", lambda s: (
        any((inp.get("type") in (None, "text")) and not inp.has_attr("list") and labeled(s, inp) for inp in s.find_all("input"))
        and all(any(inp.get("type") == kind and labeled(s, inp) for inp in s.find_all("input")) for kind in ("email", "date", "color", "range"))
    )),
    ("A datalist offers at least three courses", lambda s: any(
        inp.get("list") and s.find("datalist", id=inp.get("list")) is not None and len(s.find("datalist", id=inp.get("list")).find_all("option")) >= 3
        for inp in s.find_all("input")
    )),
    ("Three FAQ items exist and one starts open", lambda s: len(s.find_all("details")) >= 3 and any(item.has_attr("open") for item in s.find_all("details")) and all(item.summary and item.summary.get_text(strip=True) for item in s.find_all("details"))),
    ("A button opens a popover", lambda s: any(
        s.find(id=btn.get("popovertarget", "")) is not None and s.find(id=btn.get("popovertarget", "")).has_attr("popover")
        for btn in s.find_all("button") if btn.get("popovertarget")
    )),
    ("A dialog has open and close buttons", lambda s: any(
        dlg.get("id")
        and any(btn.get("commandfor") == dlg.get("id") and btn.get("command") == "show-modal" for btn in s.find_all("button"))
        and any(btn.get("commandfor") == dlg.get("id") and btn.get("command") == "close" for btn in dlg.find_all("button"))
        for dlg in s.find_all("dialog")
    )),
    ("Progress and meter each have a value", lambda s: s.progress is not None and s.progress.has_attr("value") and s.meter is not None and s.meter.has_attr("value")),
    ("The bio is editable", lambda s: any((el.get("contenteditable") or "").lower() == "true" for el in s.find_all(True))),
    ("The page has no CSS or JavaScript", lambda s: s.find("style") is None and s.find("script") is None and not any(el.has_attr("style") for el in s.find_all(True))),
])

## Finish & Submit

1. Rerun the final tests until all checks pass.
2. Download the page and open it in Chrome. Confirm the popover, the dialog, the open FAQ item, and the editable bio.
3. Save your notebook copy in Google Drive.
4. Submit the notebook or the HTML file when your instructor asks for it. Assignments are in [Google Classroom](https://classroom.google.com/w/ODkxNzk1OTQwMDY3/t/all).
5. Continue with the numbered exercises in the [GitHub HTML Practice repository](https://github.com/morbargig/html-practice).

**Self-check**
- Every form control has a label.
- The datalist still accepts a course that is not in the list.
- Each FAQ opens and closes from the keyboard.
- The popover opens from its button.
- The dialog opens and closes in a current browser.
- `<progress>` shows task completion, and `<meter>` shows a measurement.
- The page contains HTML only.

**Reflection:** Which native HTML feature surprised you most? When would a page still need JavaScript?

**Important:** These checks test the structure of the page. Passing them means the required elements and attributes are present. Try the interactive controls in a desktop browser before you submit.

### Further reading
- [MDN: The Input element](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Elements/input)
- [MDN: datalist](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Elements/datalist)
- [MDN: dialog](https://developer.mozilla.org/en-US/docs/Web/HTML/Reference/Elements/dialog)
- [MDN: Popover API](https://developer.mozilla.org/en-US/docs/Web/API/Popover_API)
- [GitHub HTML Practice](https://github.com/morbargig/html-practice)